# Laboratory Activity 1: Data Cleaning Pipeline
## Water Potability and Chemical Safety Assessment

**Course:** DS311 - Exploratory Data Analysis  
**Domain:** Water Quality & Public Health  
**Dataset:** `water_potability.csv` (3,276 water samples, 10 features)  

---

### Project Overview
Clean drinking water is essential for public health. In water quality monitoring, we collect measurements like pH, mineral levels, and disinfectants to determine if water is safe to drink (`Potability = 1`) or unsafe (`Potability = 0`).

However, real-world datasets often have data quality issues:
1. **Missing Test Results:** Some water samples were not tested for every chemical (especially Sulfate, pH, and Trihalomethanes).
2. **Sensor Errors:** Faulty or uncalibrated probes can produce impossible values (such as pH readings outside 0 to 14).
3. **Extreme Values:** High mineral concentrations occur naturally in some water sources, while others may be sensor spikes.

### Core Investigation Question
> *How can we systematically detect, audit, and clean sensor probe anomalies, missing chemical tests, and mineral concentration outliers to ensure that water safety classification strictly adheres to World Health Organization (WHO) safety standards?*

**Simplified Version (Plain Language):**
> *How can we fix sensor errors and fill in missing test data so we can accurately determine whether water is safe to drink?*

### Main Goal
Clean, fix, and prepare the water quality dataset using clear rules so the data is accurate, complete, and ready for exploration and modeling.


---
### Data Preparation: Key Concepts

In our lessons, preparing data involves different steps that work together:

| Stage | Goal | What We Do Here |
| :--- | :--- | :--- |
| **Data Wrangling** | The overall process of taking raw data and turning it into a usable format. | Managing the entire workflow from loading data to saving the cleaned file. |
| **Data Cleaning** | Finding and fixing errors, missing values, duplicates, and invalid readings. | Fixing impossible pH values, filling missing numbers, checking duplicates, and checking outliers. |
| **Data Transformation** | Changing the format or scale of data without adding new information. | Rounding decimal numbers and ensuring the target is a clean integer. |
| **Feature Engineering** | Creating new variables from existing ones to help analysis or models. | (Optional for downstream modeling). |

> **Core Rule:** *We do not change data just because it looks unusual. We first check the data, understand why it looks that way, and choose the best way to handle it.*

---
### Dataset Variables and Guidelines

| Variable | Type | Unit | Description & Guidelines |
| :--- | :--- | :--- | :--- |
| `ph` | `float64` | 0 - 14 | Measures how acidic or basic water is (WHO safe range: 6.5 - 8.5) |
| `Hardness` | `float64` | mg/L | Amount of calcium and magnesium in water |
| `Solids` | `float64` | ppm | Total Dissolved Solids (TDS); total mineral content |
| `Chloramines` | `float64` | ppm | Disinfectant added to kill bacteria (Safe limit: <= 4 ppm) |
| `Sulfate` | `float64` | mg/L | Natural minerals from soil and rock (Guideline: <= 250 mg/L) |
| `Conductivity` | `float64` | uS/cm | How well water conducts electricity (indicates dissolved salts) |
| `Organic_carbon` | `float64` | ppm | Level of organic matter in the water |
| `Trihalomethanes` | `float64` | ug/L | Byproducts from chlorine treatment (Safe limit: <= 80 ug/L) |
| `Turbidity` | `float64` | NTU | Cloudiness or haziness of the water (Safe limit: <= 5 NTU) |
| `Potability` | `int64` | 0 or 1 | Target: `0` = Not Potable (Unsafe), `1` = Potable (Safe to drink) |

---
## 1. Environment Setup and Library Imports

We load standard Python tools for data handling and plotting:
- `pandas` and `numpy`: For handling tables and math operations.
- `matplotlib` and `seaborn`: For charts and graphs.
- `missingno`: For visualizing missing data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

# Plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Libraries loaded successfully.")

---
## 2. Raw Data Ingestion and Initial Profiling

We start by loading the dataset and taking a first look at the rows, columns, and data types.

In [ ]:
# Load dataset
raw_data_path = "water_potability.csv"
df_raw = pd.read_csv(raw_data_path)

print(f"Dataset Size: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
df_raw.head()

### Checking Data Types and Non-Null Counts

We use `df.info()` to see if columns have the right data types and check where values might be missing.

In [ ]:
df_raw.info()

### Statistical Summary

We use `.describe()` to check the average, minimum, maximum, and spread for each column.

In [ ]:
df_raw.describe().T

---
### Checking for Duplicate Rows

Duplicate rows can skew results and give false importance to repeated samples. We check for exact duplicate rows.

In [ ]:
duplicates = df_raw.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    df_raw = df_raw.drop_duplicates().reset_index(drop=True)
    print("Duplicates removed.")
else:
    print("No duplicate rows found.")

---
### Checking Missing Values

Let us see which columns have missing values and what percentage is missing.

In [ ]:
# Count missing values per column
missing_table = pd.DataFrame({
    'Missing Count': df_raw.isna().sum(),
    'Missing Percentage (%)': (df_raw.isna().sum() / len(df_raw) * 100).round(2)
})
missing_columns = missing_table[missing_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)
missing_columns

In [ ]:
# Visualizing missing values
plt.figure(figsize=(10, 4))
msno.matrix(df_raw, figsize=(10, 4), color=(0.2, 0.4, 0.6), fontsize=10)
plt.title('Missing Values in Raw Dataset', fontsize=13, fontweight='bold', pad=15)
plt.show()

In [ ]:
# Bar plot of missing percentages
plt.figure(figsize=(8, 4))
ax = sns.barplot(
    x=missing_columns.index,
    y=missing_columns['Missing Percentage (%)'],
    palette='crest'
)
plt.title('Percentage of Missing Values per Feature', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Missing Percentage (%)')
plt.xlabel('Water Feature')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, xytext=(0, 3), textcoords='offset points')
plt.ylim(0, 30)
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning Step 1: Checking and Enforcing pH Limits (0 to 14)

### Why We Do This:
- By definition, the pH scale ranges strictly from **0 to 14**.
- Any pH value below 0 or above 14 is physically impossible in natural water and indicates a broken or uncalibrated sensor.
- We turn any out-of-range value into `NaN` so it can be filled properly.

In [ ]:
df = df_raw.copy()

# Find any pH values outside 0 to 14
invalid_ph = (df["ph"] < 0) | (df["ph"] > 14)
print(f"Impossible pH readings detected: {invalid_ph.sum()}")

# Set invalid pH to NaN
df.loc[invalid_ph, "ph"] = np.nan
print("pH boundary check complete: All values are within 0 to 14.")

---
## 4. Data Cleaning Step 2: Filling Missing Values by Water Safety Group

### Why We Use Class-Conditional Median Imputation:
1. **Why not just drop rows (`dropna()`)?**
   - Sulfate is missing in ~23.8% of rows, pH in ~15.0%, and Trihalomethanes in ~4.95%.
   - Overall, **1,265 rows (38.6% of the dataset)** have at least one missing value.
   - If we simply delete these rows, we lose more than one-third of our data. That weakens our analysis and creates biased results.
2. **Why use the Median instead of the Mean?**
   - Water chemical levels can have extreme spikes. The mean (average) gets pulled by extreme numbers, while the **median** (middle value) is stable and realistic.
3. **Why group by `Potability`?**
   - Safe water (`Potability = 1`) and unsafe water (`Potability = 0`) naturally have different chemical levels.
   - Filling missing values using the median of each group keeps safe and unsafe water profiles distinct and realistic.

In [ ]:
# Look at distributions before filling missing values
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
impute_columns = ["ph", "Sulfate", "Trihalomethanes"]

for i, col in enumerate(impute_columns):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#3498db')
    axes[i].set_title(f'Distribution: {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Fill missing values using the median of each Potability group
for col in impute_columns:
    med_potable = df[df['Potability'] == 1][col].median()
    med_non_potable = df[df['Potability'] == 0][col].median()
    print(f"{col} -> Median for Potable (1): {med_potable:.3f} | Non-Potable (0): {med_non_potable:.3f}")
    
    df[col] = df.groupby("Potability")[col].transform(
        lambda group: group.fillna(group.median())
    )

# Safety check: make sure no missing values remain
for col in df.columns:
    if df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"\nRemaining missing values in dataset: {df.isna().sum().sum()}")

In [ ]:
# Look at distributions after filling missing values
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(impute_columns):
    sns.kdeplot(data=df, x=col, hue='Potability', common_norm=False, ax=axes[i], palette=['#e74c3c', '#2ecc71'])
    axes[i].set_title(f'Post-Imputation: {col}')
plt.tight_layout()
plt.show()

---
## 5. Data Cleaning Step 3: Checking and Handling Outliers (Total Dissolved Solids)

### Understanding Outliers in Environmental Data:
- **What are Solids?** Total Dissolved Solids (`Solids`) measure total minerals and salts in the water.
- **Natural Variation vs. Broken Sensors:**
  - Tap water usually has 100 to 1,000 ppm of solids.
  - Deep mineral wells and brackish groundwater naturally have 10,000 to 30,000 ppm.
- **Choosing the Right Cutoff:**
  - A standard boxplot rule ($1.5 \times \text{IQR}$) would delete 47 valid mineral-rich water samples.
  - Instead, we use an **extreme boundary ($3.0 \times \text{IQR}$)**. This keeps natural mineral-rich water while removing any impossible sensor spikes (> 60,000 ppm).

In [ ]:
# Calculate standard (1.5*IQR) and extreme (3.0*IQR) boundaries for Solids
q1 = df["Solids"].quantile(0.25)
q3 = df["Solids"].quantile(0.75)
iqr = q3 - q1
mild_limit = q3 + 1.5 * iqr
extreme_limit = q3 + 3.0 * iqr

mild_outliers = (df["Solids"] > mild_limit).sum()
extreme_outliers = (df["Solids"] > extreme_limit).sum()

print(f"Solids Q1: {q1:.2f} ppm | Q3: {q3:.2f} ppm | IQR: {iqr:.2f} ppm")
print(f"Standard Cutoff (1.5 * IQR = {mild_limit:.2f} ppm): {mild_outliers} samples detected")
print(f"Extreme Cutoff  (3.0 * IQR = {extreme_limit:.2f} ppm): {extreme_outliers} samples detected")

In [ ]:
# Boxplot showing the distribution and both boundaries
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['Solids'], color='#a8d8ea')
plt.axvline(mild_limit, color='orange', linestyle=':', label=f'1.5*IQR Standard Fence ({mild_limit:.1f} ppm)')
plt.axvline(extreme_limit, color='red', linestyle='--', label=f'3.0*IQR Extreme Fence ({extreme_limit:.1f} ppm)')
plt.title('Total Dissolved Solids (ppm) Outlier Analysis', fontsize=13, fontweight='bold')
plt.xlabel('Solids (ppm)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Keep only rows that fall within the extreme boundary
df_cleaned = df[df["Solids"] <= extreme_limit].reset_index(drop=True)
print(f"Rows before: {len(df):,}")
print(f"Rows after:  {len(df_cleaned):,}")

---
## 6. Data Cleaning Step 4: Formatting and Consistency

### Final Adjustments:
1. **Integer Target:** Make sure `Potability` is stored as an integer (`0` or `1`).
2. **Clean Decimal Precision:** Round continuous measurements to **3 decimal places** so the numbers are clean and consistent.
3. **Verification:** Check that there are 0 missing values and no duplicates.

In [ ]:
# 1. Ensure Potability is int
df_cleaned["Potability"] = df_cleaned["Potability"].astype(int)

# 2. Round continuous numbers to 3 decimal places
continuous_cols = [col for col in df_cleaned.columns if col != "Potability"]
for col in continuous_cols:
    df_cleaned[col] = df_cleaned[col].round(3)

# 3. Check and confirm
assert df_cleaned.isna().sum().sum() == 0, "There are still missing values!"
assert df_cleaned['Potability'].isin([0, 1]).all(), "Potability has invalid labels!"
assert df_cleaned.duplicated().sum() == 0, "Duplicate rows detected!"

print("All checks passed successfully.")
df_cleaned.info()

---
## 7. Data Cleaning Step 5: Export Cleaned Dataset

We save the finalized clean data to `water_potability_cleaned.csv`.

In [ ]:
output_csv_path = "water_potability_cleaned.csv"
df_cleaned.to_csv(output_csv_path, index=False)
print(f"Cleaned dataset saved to: {output_csv_path}")

---
## 8. Before vs. After Data Cleaning Evaluation

Let us compare the original raw dataset against our cleaned dataset to see the changes made.

In [ ]:
# Comparison table
comparison_table = pd.DataFrame({
    "Check / Metric": [
        "Total Rows",
        "Total Missing Values",
        "Missing Sulfate Rows",
        "Missing pH Rows",
        "Missing Trihalomethanes Rows",
        "Invalid pH (outside 0-14)",
        "Duplicate Rows",
        "Max Solids (ppm)",
        "Potable Water Ratio (%)",
        "Status"
    ],
    "Raw Dataset": [
        f"{len(df_raw):,}",
        f"{df_raw.isna().sum().sum():,}",
        f"{df_raw['Sulfate'].isna().sum()} ({df_raw['Sulfate'].isna().mean()*100:.2f}%)",
        f"{df_raw['ph'].isna().sum()} ({df_raw['ph'].isna().mean()*100:.2f}%)",
        f"{df_raw['Trihalomethanes'].isna().sum()} ({df_raw['Trihalomethanes'].isna().mean()*100:.2f}%)",
        f"{((df_raw['ph'] < 0) | (df_raw['ph'] > 14)).sum()}",
        f"{df_raw.duplicated().sum()}",
        f"{df_raw['Solids'].max():.2f} ppm",
        f"{df_raw['Potability'].sum()} ({df_raw['Potability'].mean()*100:.2f}%)",
        "38.6% Incomplete Rows"
    ],
    "Cleaned Dataset": [
        f"{len(df_cleaned):,}",
        "0",
        "0 (0.00%)",
        "0 (0.00%)",
        "0 (0.00%)",
        "0 (Enforced 0 <= pH <= 14)",
        "0",
        f"{df_cleaned['Solids'].max():.2f} ppm",
        f"{df_cleaned['Potability'].sum()} ({df_cleaned['Potability'].mean()*100:.2f}%)",
        "Complete & Ready for EDA"
    ]
})

comparison_table

In [ ]:
# Visual check: Missing values before vs after
fig, ax = plt.subplots(figsize=(6, 4))
categories = ['Raw Dataset', 'Cleaned Dataset']
missing_totals = [df_raw.isna().sum().sum(), df_cleaned.isna().sum().sum()]

bars = ax.bar(categories, missing_totals, color=['#e74c3c', '#2ecc71'], width=0.45)
plt.title('Total Missing Values: Before vs. After Cleaning', fontsize=13, fontweight='bold', pad=12)
plt.ylabel('Count of Missing Values')
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 25, f"{int(yval):,}", ha='center', va='bottom', fontweight='bold')
plt.ylim(0, max(missing_totals) + 200)
plt.tight_layout()
plt.show()

---
## 9. Presentation and Defense Guide

Clear, simple answers to explain our cleaning decisions during presentation:

### Question 1: Why did you fill missing values instead of deleting rows?
> **Answer:** Over 38% of the dataset (1,265 rows) had at least one missing chemical test. If we deleted those rows with `dropna()`, we would lose more than one-third of our data. Filling them keeps all 3,276 rows of data for analysis.

### Question 2: Why did you use the median instead of the mean?
> **Answer:** Environmental chemical levels often have extreme high or low values. The mean gets pulled by extreme numbers, but the median represents the typical center and is not affected by outliers.

### Question 3: Why did you fill missing values separately for Potable and Non-Potable water?
> **Answer:** Safe drinking water and unsafe water naturally have different chemical levels. Grouping by Potability ensures that we fill missing values with realistic numbers for each type of water.

### Question 4: Why must pH be between 0 and 14?
> **Answer:** The pH scale is scientifically defined from 0 (very acidic) to 14 (very basic). Any value outside this range is a faulty sensor reading, not real water chemistry.

### Question 5: Why didn't you remove all high Total Dissolved Solids as outliers?
> **Answer:** Natural mineral water and groundwater can legitimately have high mineral levels (10,000 to 30,000 ppm). Removing them with a standard 1.5x IQR rule would delete valid natural water samples. Using an extreme 3.0x IQR cutoff keeps real water samples and only removes unrealistic sensor errors.